# Simulation and Data Generation

This notebook demonstrates how to:
1. Simulate optimal replacement behavior
2. Generate synthetic datasets
3. Analyze replacement patterns
4. Create Rust (1987) style bus data

These tools are essential for:
- Testing estimation procedures
- Understanding model behavior
- Validating implementations

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

from src.simulation import (
    ReplacementSimulator,
    create_rust_dataset,
    analyze_replacement_patterns
)
from src.transitions import DiscreteTransition
from src.cost_functions import LinearCost
from src.dp import DynamicProgram

## 1. Setup: Solve for Optimal Policy

First, we need to solve the dynamic program to get the optimal policy.

In [ ]:
# Parameters
n_states = 90
discount_factor = 0.9999  # Monthly discount
replacement_cost = 11.7   # Thousand dollars

# Cost function: linear
cost_fn = LinearCost(theta=np.array([0.0, 0.001]))
state_grid = np.arange(n_states)
maintenance_costs = cost_fn.evaluate(state_grid)

# Transition model
trans_model = DiscreteTransition(n_states, transition_probs=np.array([0.349, 0.639, 0.012]))
P = trans_model.get_transition_matrix()

# Solve DP
dp = DynamicProgram(n_states, discount_factor, P, maintenance_costs, replacement_cost)
result = dp.value_iteration(verbose=True)

print(f"\nOptimal policy computed in {result['iterations']} iterations")

## 2. Simulate Single Agent Trajectory

Simulate one agent following the optimal policy.

In [ ]:
# Create simulator
simulator = ReplacementSimulator(
    n_states=n_states,
    transition_matrix=P,
    policy=result['policy'],
    random_state=42
)

# Simulate 200 periods
df_single = simulator.simulate_trajectory(n_periods=200, initial_state=0)

print(df_single.head(20))
print(f"\nReplacement rate: {df_single['decision'].mean():.4f}")

In [ ]:
# Visualize trajectory
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# State trajectory
axes[0].plot(df_single['period'], df_single['state'], linewidth=2)
replacement_periods = df_single[df_single['decision'] == 1]['period']
axes[0].scatter(replacement_periods, [0]*len(replacement_periods), 
               color='red', s=100, marker='v', label='Replacement', zorder=5)
axes[0].set_ylabel('State (Mileage)', fontsize=12)
axes[0].set_title('State Evolution Over Time', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decisions
axes[1].fill_between(df_single['period'], 0, df_single['decision'], 
                      alpha=0.5, label='Replace')
axes[1].set_xlabel('Period', fontsize=12)
axes[1].set_ylabel('Decision', fontsize=12)
axes[1].set_title('Replacement Decisions', fontsize=14)
axes[1].set_ylim(-0.1, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Simulate Panel Data

Generate data for multiple agents (buses).

In [ ]:
# Simulate panel with 50 buses over 100 periods each
df_panel = simulator.simulate_panel(n_agents=50, n_periods=100)

print(f"Total observations: {len(df_panel)}")
print(f"Number of agents: {df_panel['agent'].nunique()}")
print(f"\nFirst few rows:")
print(df_panel.head(10))

In [ ]:
# Analyze patterns
patterns = analyze_replacement_patterns(df_panel)

print("Summary Statistics:")
print(f"  Replacement rate: {patterns['replacement_rate']:.4f}")
print(f"  Mean time between replacements: {patterns.get('mean_time_between_replacements', 'N/A')}")
print(f"  Median time between replacements: {patterns.get('median_time_between_replacements', 'N/A')}")

In [ ]:
# Visualize panel patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# State distribution
axes[0, 0].hist(df_panel['state'], bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('State (Mileage)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('State Distribution', fontsize=12)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Hazard rate
hazard = patterns['hazard_rate']
axes[0, 1].plot(hazard.index, hazard.values, linewidth=2, marker='o', markersize=4)
axes[0, 1].set_xlabel('State (Mileage)', fontsize=11)
axes[0, 1].set_ylabel('Replacement Probability', fontsize=11)
axes[0, 1].set_title('Hazard Rate by State', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# Sample trajectories (first 5 agents)
for agent in range(5):
    agent_data = df_panel[df_panel['agent'] == agent]
    axes[1, 0].plot(agent_data['period'], agent_data['state'], 
                   alpha=0.6, linewidth=1.5, label=f'Agent {agent}')
axes[1, 0].set_xlabel('Period', fontsize=11)
axes[1, 0].set_ylabel('State (Mileage)', fontsize=11)
axes[1, 0].set_title('Sample Trajectories (5 Agents)', fontsize=12)
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Replacement frequency by agent
replacement_counts = df_panel.groupby('agent')['decision'].sum()
axes[1, 1].hist(replacement_counts, bins=20, alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Number of Replacements', fontsize=11)
axes[1, 1].set_ylabel('Number of Agents', fontsize=11)
axes[1, 1].set_title('Replacements per Agent', fontsize=12)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Generate Rust (1987) Style Dataset

Create a dataset mimicking the original bus engine replacement data.

In [ ]:
# Generate Rust-style dataset
df_rust = create_rust_dataset(sample_size=5000, n_states=90, random_state=42)

print(f"Dataset size: {len(df_rust)} observations")
print(f"Number of buses: {df_rust['bus_id'].nunique()}")
print(f"\nFirst few rows:")
print(df_rust.head(15))

In [ ]:
# Descriptive statistics
print("Descriptive Statistics:")
print(df_rust[['mileage_bin', 'decision']].describe())

print(f"\nReplacement rate: {df_rust['decision'].mean():.4f}")
print(f"Mean mileage: {df_rust['mileage_bin'].mean():.2f} thousand miles")
print(f"Median mileage: {df_rust['mileage_bin'].median():.2f} thousand miles")

In [ ]:
# Visualize Rust dataset
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mileage distribution
axes[0, 0].hist(df_rust['mileage_bin'], bins=30, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(df_rust['mileage_bin'].mean(), color='r', 
                  linestyle='--', linewidth=2, label='Mean')
axes[0, 0].set_xlabel('Mileage Bin', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Mileage Distribution', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Replacement probability by mileage
replacement_prob = df_rust.groupby('mileage_bin')['decision'].agg(['mean', 'count'])
# Filter out bins with few observations
replacement_prob = replacement_prob[replacement_prob['count'] >= 5]
axes[0, 1].scatter(replacement_prob.index, replacement_prob['mean'], 
                  s=replacement_prob['count'], alpha=0.6)
axes[0, 1].set_xlabel('Mileage Bin', fontsize=11)
axes[0, 1].set_ylabel('Replacement Probability', fontsize=11)
axes[0, 1].set_title('Replacement Probability by Mileage\n(bubble size = sample size)', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

# Decisions over time (sample buses)
sample_buses = df_rust['bus_id'].unique()[:10]
sample_data = df_rust[df_rust['bus_id'].isin(sample_buses)]
for bus_id in sample_buses:
    bus_data = sample_data[sample_data['bus_id'] == bus_id]
    axes[1, 0].plot(bus_data['period'], bus_data['mileage_bin'], alpha=0.5, linewidth=1)
axes[1, 0].set_xlabel('Period', fontsize=11)
axes[1, 0].set_ylabel('Mileage Bin', fontsize=11)
axes[1, 0].set_title('Mileage Trajectories (Sample of 10 Buses)', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

# State transitions
# Look at increments when not replacing
no_replace = df_rust[df_rust['decision'] == 0].copy()
no_replace = no_replace.sort_values(['bus_id', 'period'])
no_replace['mileage_increment'] = no_replace.groupby('bus_id')['mileage_bin'].diff()
increments = no_replace['mileage_increment'].dropna()
increments = increments[increments >= 0]  # Keep only forward transitions
axes[1, 1].hist(increments, bins=range(0, 10), alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Mileage Increment', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Mileage Increments (When Not Replacing)', fontsize=12)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Save Dataset for Later Use

Save the generated dataset for use in estimation notebooks.

In [ ]:
# Save to CSV
output_path = '../data/synthetic_rust_data.csv'
df_rust.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")
print(f"Shape: {df_rust.shape}")
print(f"Columns: {list(df_rust.columns)}")

## Summary

In this notebook, we:
1. ✓ Simulated optimal replacement behavior
2. ✓ Generated panel datasets
3. ✓ Analyzed replacement patterns
4. ✓ Created Rust (1987) style data

**Key Takeaways:**
- Optimal policy leads to threshold behavior
- State distribution reflects equilibrium dynamics
- Hazard rate increases with mileage
- Panel data captures heterogeneity across agents

**Next:** In the next notebook, we'll use this simulated data to estimate parameters using NFXP.